## Is it a bird?

In [ ]:
# Run this notebook with the project .venv kernel (see ../../README.md next to this notebook).
# Live image search and the first download of pretrained weights need internet access.

In [ ]:
# The project environment supplies the packages used throughout the lesson.
import sys
print(sys.executable)

## Step 1: Download images of birds and non-birds

In [ ]:
# If search or image hosts are unavailable, follow "Use local photos" in ../../README.md
# and change this to True. The rest of the lesson can then run with your photos.
use_local_photos = False

In [ ]:
from ddgs import DDGS
from fastcore.all import *

def search_images(keywords, max_images=200):
    return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')

In [ ]:
if not use_local_photos:
    urls = search_images('bird photos', max_images=1)
    display(urls[0])

In [ ]:
from fastdownload import download_url
dest = 'bird.jpg'
if not use_local_photos:
    download_url(urls[0], dest, show_progress=False, timeout=15)

from fastai.vision.all import *
im = Image.open(dest)
im.to_thumb(256,256)

In [ ]:
if not use_local_photos:
    download_url(search_images('forest photos', max_images=1)[0], 'forest.jpg', show_progress=False, timeout=15)
Image.open('forest.jpg').to_thumb(256,256)

In [ ]:
searches = 'forest','bird'
path = Path('bird_or_not')
from time import sleep

for o in searches:
    dest = (path/o)
    if not use_local_photos:
        dest.mkdir(exist_ok=True, parents=True)
        download_images(dest, urls=search_images(f'{o} photo'))
        sleep(10)  # Pause between searches to avoid over-loading server
        download_images(dest, urls=search_images(f'{o} sun photo'))
        sleep(10)
        download_images(dest, urls=search_images(f'{o} shade photo'))
        sleep(10)
    resize_images(path/o, max_size=400, dest=path/o)

## Step 2: Train our model

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path)

dls.show_batch(max_n=6)

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

## Step 3: Use our model (and build your own!)

In [ ]:
is_bird,_,probs = learn.predict(PILImage.create('bird.jpg'))
print(f"This is a: {is_bird}.")
print(f"Probability it's a bird: {probs[0]:.4f}\nProbability it's a forest image: {probs[1]:.4f}")